# Gathering raw data

## First we must import the necessary libraries

In [1]:
import json
import time
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

## Configuration

All three APIs ask for a descriptive `User-Agent` with contact info, so **replace `CONTACT` with your own email before running**.
- Wikipedia: send a descriptive UA, make requests one at a time, and use `maxlag`.
- MusicBrainz: at most 1 request per second.
- iTunes: about 20 requests per minute.

In [2]:
CONTACT = "example@contact.com"  # replace with your contact email
HEADERS = {"User-Agent": f"song-duration-analysis/0.1 ( {CONTACT} )"}

RAW_DIR = Path("../data/raw/")
PROCESSED_DIR = Path("../data/processed/")

YEARS = range(1970, 2026)  # Billboard Year-End charts, 1970 through 2025
SONGS_PER_YEAR = 2  # each year-end chart contributes its top 2 songs to its decade

WIKI_FILE = RAW_DIR / "wikipedia_year_end_hot100.json"
MB_FILE = RAW_DIR / "musicbrainz_recordings.json"
ITUNES_FILE = RAW_DIR / "itunes_tracks.json"
DECADE_FILE = PROCESSED_DIR / "decade_top_songs.json"

## Helpers

`get_json` sends the shared headers, retries with backoff when an API returns 429 or 503, and sleeps afterward to respect each API's rate limit.
Each raw file is fetched once. If it already exists, it's loaded instead, so raw data is never overwritten.

In [3]:
def get_json(url, params, delay, max_retries=5):
    for attempt in range(max_retries):
        response = requests.get(url, params=params, headers=HEADERS, timeout=30)
        if response.status_code in (429, 503):
            wait = int(response.headers.get("Retry-After", 2 ** (attempt + 1)))
            print(f"  {response.status_code} from {url}, retrying in {wait}s")
            time.sleep(wait)
            continue
        response.raise_for_status()
        time.sleep(delay)
        return response.json()
    response.raise_for_status()


def now():
    return datetime.now(timezone.utc).isoformat()


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## Step 1: Billboard Year-End Hot 100 charts (Wikipedia)

Neither MusicBrainz nor iTunes provides chart rankings, so the song list comes from the Wikipedia pages "Billboard Year-End Hot 100 singles of YYYY", read through the [MediaWiki API](https://www.mediawiki.org/wiki/API:Main_page).
Each API response is stored unaltered.

In [4]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

if WIKI_FILE.exists():
    wiki_raw = load_json(WIKI_FILE)
    print(f"Loaded {len(wiki_raw)} years from {WIKI_FILE}")
else:
    wiki_raw = {}
    for year in YEARS:
        params = {
            "action": "parse",
            "page": f"Billboard_Year-End_Hot_100_singles_of_{year}",
            "prop": "text|revid",
            "format": "json",
            "formatversion": 2,
            "maxlag": 5,
        }
        wiki_raw[str(year)] = {
            "retrieved_at": now(),
            "url": WIKI_API,
            "params": params,
            "response": get_json(WIKI_API, params, delay=1),
        }
        print(f"Fetched {year}")
    save_json(wiki_raw, WIKI_FILE)
    print(f"Saved {len(wiki_raw)} years to {WIKI_FILE}")

Fetched 1970
Fetched 1971
Fetched 1972
Fetched 1973
Fetched 1974
Fetched 1975
Fetched 1976
Fetched 1977
Fetched 1978
Fetched 1979
Fetched 1980
Fetched 1981
Fetched 1982
Fetched 1983
Fetched 1984
Fetched 1985
Fetched 1986
Fetched 1987
Fetched 1988
Fetched 1989
Fetched 1990
Fetched 1991
Fetched 1992
Fetched 1993
Fetched 1994
Fetched 1995
Fetched 1996
Fetched 1997
Fetched 1998
Fetched 1999
Fetched 2000
Fetched 2001
Fetched 2002
Fetched 2003
Fetched 2004
Fetched 2005
Fetched 2006
Fetched 2007
Fetched 2008
Fetched 2009
Fetched 2010
Fetched 2011
Fetched 2012
Fetched 2013
Fetched 2014
Fetched 2015
Fetched 2016
Fetched 2017
Fetched 2018
Fetched 2019
Fetched 2020
Fetched 2021
Fetched 2022
Fetched 2023
Fetched 2024
Fetched 2025
Saved 56 years to ..\data\raw\wikipedia_year_end_hot100.json


## Step 2: Parse the chart tables

The chart is the page table with a `Title` column. The rank column header is either `No.` or `№` depending on the year, so columns are selected by position.
Titles and artists are kept exactly as Wikipedia shows them, including the quotation marks.

In [5]:
rows = []
for year, entry in wiki_raw.items():
    html = entry["response"]["parse"]["text"]
    chart = next(t for t in pd.read_html(StringIO(html)) if "Title" in t.columns)
    for rank, title, artist in chart.iloc[:, :3].itertuples(index=False):
        rows.append({"year": int(year), "rank": int(rank), "title": title, "artist": artist})

charts = pd.DataFrame(rows)
assert charts.groupby("year").size().eq(100).all(), "every year should have 100 ranked songs"
print(f"{charts['year'].nunique()} years, {len(charts)} chart entries")
charts.head()

56 years, 5600 chart entries


,year,rank,title,artist
0,1970,1,"""Bridge Over Troubled Water""",Simon & Garfunkel
1,1970,2,"""(They Long to Be) Close to You""",The Carpenters
2,1970,3,"""American Woman""",The Guess Who
3,1970,4,"""Raindrops Keep Fallin' on My Head""",B.J. Thomas
4,1970,5,"""War""",Edwin Starr


## Step 3: Select the top songs per decade

There is no consistent decade-end chart, so each year contributes its top 2 songs. A full decade gets 20 songs, and the 2020s (2020 through 2025) get 12.
Every decade is selected the same way. Accumulating points across years instead would favor songs that span two chart years, which happens much more often from the 1990s on.

If a song was already picked earlier in the same decade, the next-highest new song from that year is used instead.
Songs are matched across years by exact title and artist text. This selection is derived data, so it goes to `data/processed/`.

In [6]:
selected = []
for year, chart in charts.groupby("year"):
    decade = year // 10 * 10
    seen = {(s["title"], s["artist"]) for s in selected if s["decade"] == decade}
    new_songs = [row for row in chart.itertuples() if (row.title, row.artist) not in seen]
    for row in new_songs[:SONGS_PER_YEAR]:
        selected.append({"decade": decade, "year": year, "year_rank": row.rank, "title": row.title, "artist": row.artist})

decade_top = pd.DataFrame(selected)
save_json(decade_top.to_dict(orient="records"), DECADE_FILE)
print(decade_top.groupby("decade").size())
decade_top.head(20)

decade
1970    20
1980    20
1990    20
2000    20
2010    20
2020    12
dtype: int64


,decade,year,year_rank,title,artist
0,1970,1970,1,"""Bridge Over Troubled Water""",Simon & Garfunkel
1,1970,1970,2,"""(They Long to Be) Close to You""",The Carpenters
2,1970,1971,1,"""Joy to the World""",Three Dog Night
3,1970,1971,2,"""Maggie May""/""Reason to Believe""",Rod Stewart
4,1970,1972,1,"""The First Time Ever I Saw Your Face""",Roberta Flack
5,1970,1972,2,"""Alone Again (Naturally)""",Gilbert O'Sullivan
6,1970,1973,1,"""Tie a Yellow Ribbon Round the Ole Oak Tree""",Tony Orlando and Dawn
7,1970,1973,2,"""Bad, Bad Leroy Brown""",Jim Croce
8,1970,1974,1,"""The Way We Were""",Barbra Streisand
9,1970,1974,2,"""Seasons in the Sun""",Terry Jacks


## Step 4: MusicBrainz recording search

For each song, search MusicBrainz [recordings](https://musicbrainz.org/doc/MusicBrainz_API/Search) by title and artist, keeping the top 10 candidates.
The surrounding quotation marks are stripped from the title for the query only, and inner double quotes are escaped for Lucene. Choosing the right candidate is left to the cleaning step.

In [7]:
MB_API = "https://musicbrainz.org/ws/2/recording/"


def query_title(title):
    return title.strip('"')


def lucene_phrase(text):
    return '"' + text.replace('"', r'\"') + '"'


if MB_FILE.exists():
    mb_raw = load_json(MB_FILE)
    print(f"Loaded {len(mb_raw)} records from {MB_FILE}")
else:
    mb_raw = []
    for song in decade_top.itertuples():
        params = {
            "query": f"recording:{lucene_phrase(query_title(song.title))} AND artist:{lucene_phrase(song.artist)}",
            "limit": 10,
            "fmt": "json",
        }
        mb_raw.append({
            "decade": int(song.decade),
            "year": int(song.year),
            "year_rank": int(song.year_rank),
            "title": song.title,
            "artist": song.artist,
            "retrieved_at": now(),
            "url": MB_API,
            "params": params,
            "response": get_json(MB_API, params, delay=1.1),
        })
        print(f"{song.year} #{song.year_rank}: {song.title} - {song.artist}")
    save_json(mb_raw, MB_FILE)
    print(f"Saved {len(mb_raw)} records to {MB_FILE}")

  503 from https://musicbrainz.org/ws/2/recording/, retrying in 0s
1970 #1: "Bridge Over Troubled Water" - Simon & Garfunkel
  503 from https://musicbrainz.org/ws/2/recording/, retrying in 0s
1970 #2: "(They Long to Be) Close to You" - The Carpenters
1971 #1: "Joy to the World" - Three Dog Night
1971 #2: "Maggie May"/"Reason to Believe" - Rod Stewart
  503 from https://musicbrainz.org/ws/2/recording/, retrying in 0s
1972 #1: "The First Time Ever I Saw Your Face" - Roberta Flack
1972 #2: "Alone Again (Naturally)" - Gilbert O'Sullivan
1973 #1: "Tie a Yellow Ribbon Round the Ole Oak Tree" - Tony Orlando and Dawn
1973 #2: "Bad, Bad Leroy Brown" - Jim Croce
  503 from https://musicbrainz.org/ws/2/recording/, retrying in 0s
1974 #1: "The Way We Were" - Barbra Streisand
1974 #2: "Seasons in the Sun" - Terry Jacks
  503 from https://musicbrainz.org/ws/2/recording/, retrying in 0s
1975 #1: "Love Will Keep Us Together" - Captain & Tennille
1975 #2: "Rhinestone Cowboy" - Glen Campbell
  503 from 

## Step 5: iTunes Search API

Search the US iTunes store for each song. `trackTimeMillis` in the response is the track duration.
Apple limits this API to about 20 requests per minute, so calls are spaced about 3 seconds apart.

In [8]:
ITUNES_API = "https://itunes.apple.com/search"

if ITUNES_FILE.exists():
    itunes_raw = load_json(ITUNES_FILE)
    print(f"Loaded {len(itunes_raw)} records from {ITUNES_FILE}")
else:
    itunes_raw = []
    for song in decade_top.itertuples():
        params = {
            "term": f"{query_title(song.title)} {song.artist}",
            "media": "music",
            "entity": "song",
            "country": "US",
            "limit": 10,
        }
        itunes_raw.append({
            "decade": int(song.decade),
            "year": int(song.year),
            "year_rank": int(song.year_rank),
            "title": song.title,
            "artist": song.artist,
            "retrieved_at": now(),
            "url": ITUNES_API,
            "params": params,
            "response": get_json(ITUNES_API, params, delay=3.1),
        })
        print(f"{song.year} #{song.year_rank}: {song.title} - {song.artist}")
    save_json(itunes_raw, ITUNES_FILE)
    print(f"Saved {len(itunes_raw)} records to {ITUNES_FILE}")

1970 #1: "Bridge Over Troubled Water" - Simon & Garfunkel
1970 #2: "(They Long to Be) Close to You" - The Carpenters
1971 #1: "Joy to the World" - Three Dog Night
1971 #2: "Maggie May"/"Reason to Believe" - Rod Stewart
1972 #1: "The First Time Ever I Saw Your Face" - Roberta Flack
1972 #2: "Alone Again (Naturally)" - Gilbert O'Sullivan
1973 #1: "Tie a Yellow Ribbon Round the Ole Oak Tree" - Tony Orlando and Dawn
1973 #2: "Bad, Bad Leroy Brown" - Jim Croce
1974 #1: "The Way We Were" - Barbra Streisand
1974 #2: "Seasons in the Sun" - Terry Jacks
1975 #1: "Love Will Keep Us Together" - Captain & Tennille
1975 #2: "Rhinestone Cowboy" - Glen Campbell
1976 #1: "Silly Love Songs" - Wings
1976 #2: "Don't Go Breaking My Heart" - Elton John & Kiki Dee
1977 #1: "Tonight's the Night (Gonna Be Alright)" - Rod Stewart
1977 #2: "I Just Want to Be Your Everything" - Andy Gibb
1978 #1: "Shadow Dancing" - Andy Gibb
1978 #2: "Night Fever" - Bee Gees
1979 #1: "My Sharona" - The Knack
1979 #2: "Bad Girls" 

## Summary

The counts below show what was collected. Songs with no search results are listed for follow-up during cleaning and aren't fixed here.

In [9]:
print(f"Wikipedia year-end charts: {len(wiki_raw)} years")
print(f"Selected songs:            {len(decade_top)}")
print(f"MusicBrainz records:       {len(mb_raw)}")
print(f"iTunes records:            {len(itunes_raw)}")

mb_empty = [(r["decade"], r["title"], r["artist"]) for r in mb_raw if r["response"]["count"] == 0]
itunes_empty = [(r["decade"], r["title"], r["artist"]) for r in itunes_raw if r["response"]["resultCount"] == 0]
print(f"\nNo MusicBrainz results ({len(mb_empty)}):", *mb_empty, sep="\n  ")
print(f"\nNo iTunes results ({len(itunes_empty)}):", *itunes_empty, sep="\n  ")

Wikipedia year-end charts: 56 years
Selected songs:            112
MusicBrainz records:       112
iTunes records:            112

No MusicBrainz results (12):
  (1970, '"(They Long to Be) Close to You"', 'The Carpenters')
  (1970, '"Maggie May"/"Reason to Believe"', 'Rod Stewart')
  (1980, '"That\'s What Friends Are For"', 'Dionne and Friends (Dionne Warwick, Gladys Knight, Elton John and Stevie Wonder)')
  (1990, '"Candle in the Wind 1997" / "Something About the Way You Look Tonight"', 'Elton John')
  (1990, '"You Were Meant for Me" / "Foolish Games"', 'Jewel')
  (2000, '"Umbrella"', 'Rihanna featuring Jay-Z')
  (2010, '"Thrift Shop"', 'Macklemore & Ryan Lewis featuring Wanz')
  (2010, '"Blurred Lines"', 'Robin Thicke featuring T.I. and Pharrell Williams')
  (2010, '"Dark Horse"', 'Katy Perry featuring Juicy J')
  (2010, '"Uptown Funk"', 'Mark Ronson featuring Bruno Mars')
  (2010, '"Despacito (Remix)"', 'Luis Fonsi and Daddy Yankee featuring Justin Bieber')
  (2010, '"Old Town Road"'

## Sources

- Chart rankings: Billboard Year-End Hot 100, via Wikipedia "Billboard Year-End Hot 100 singles of YYYY" pages ([MediaWiki API](https://www.mediawiki.org/wiki/API:Main_page)). Wikipedia text is licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- Recording metadata: [MusicBrainz API](https://musicbrainz.org/doc/MusicBrainz_API). Core data is CC0.
- Track metadata: [iTunes Search API](https://performance-partners.apple.com/search-api).